# Fine-Tune Qwen3.6 (Clincoo) — LoRA / Unsloth → GGUF → Ollama

Notebook ini melatih model dasar **Qwen3.6** (model yang sama dengan `qwen3.6` di Ollama)
memakai dataset percakapan Clincoo, lalu mengekspor hasilnya ke **GGUF** supaya bisa
dijalankan langsung di Ollama lokal.

**Syarat:** runtime GPU (Colab: T4 cukup untuk QLoRA; A100 lebih cepat).
Sesuaikan `MODEL_HF` dengan nama repo Hugging Face model dasarmu.

**Dataset:** unggah `qwen3.6-dataset.jsonl` (percakapan nyata) dan
`seed-ideal.jsonl` (contoh ideal buatan model besar) ke folder `dataset/` di
sesi Colab ini. Keduanya digabung untuk latihan.

In [ ]:
# 1) Instal Unsloth (Colab)
%pip install -q unsloth
# Catatan: di luar Colab, ikuti panduan instalasi resmi unsloth.ai untuk environment kamu.

In [ ]:
# 2) Muat model dasar (4-bit / QLoRA)
from unsloth import FastLanguageModel
import torch

MODEL_HF = "unsloth/Qwen3.6-Instruct-4bit"   # <- SESUAIKAN dengan repo HF model dasar qwen3.6 kamu
MAX_SEQ = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_HF,
    max_seq_length=MAX_SEQ,
    dtype=None,            # biarkan unsloth memilih
    load_in_4bit=True,     # QLoRA: hemat VRAM
)

In [ ]:
# 3) Tambahkan adapter LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# 4) Muat dataset (gabungan percakapan nyata + seed ideal)
from datasets import load_dataset

# Unggah kedua file ini ke folder dataset/ sesi Colab:
#   dataset/qwen3.6-dataset.jsonl  (percakapan nyata hasil /api/dataset)
#   dataset/seed-ideal.jsonl       (86 contoh ideal gaya Clincoo)
ds = load_dataset("json", data_files={
    "train": ["dataset/qwen3.6-dataset.jsonl", "dataset/seed-ideal.jsonl"],
})["train"]
print(len(ds), "contoh latihan")

In [ ]:
# 5) Format chat sesuai tokenizer model, lalu latih
from unsloth.chat_templates import train_on_responses_only

def fmt(ex):
    convs = tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": convs}

ds = ds.map(fmt)
train = ds.train_test_split(test_size=0.05, seed=3407)

from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train["train"],
    eval_dataset=train["test"],
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=3,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=20,
        save_strategy="steps",
        save_steps=50,
        output_dir="outputs",
        report_to="none",
    ),
    dataset_text_field="text",
    max_seq_length=MAX_SEQ,
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
trainer.train()

In [ ]:
# 6) Tes cepat hasil latihan
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)
msgs = [{"role": "user", "content": "Apa itu CDN?"}]
ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=ids, max_new_tokens=256, temperature=0.7, top_p=0.8, top_k=20)
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

In [ ]:
# 7) Simpan adapter LoRA + model gabungan
model.save_pretrained("clincoo-qwen3.6-lora")     # adapter saja (kecil)
tokenizer.save_pretrained("clincoo-qwen3.6-lora")
model.save_pretrained_merged("clincoo-qwen3.6-merged", tokenizer)  # gabungan penuh

In [ ]:
# 8) Ekspor ke GGUF (untuk Ollama)
# Q4_K_M: kualitas/ukuran sama seperti model asli yang sekarang dipakai.
model.save_pretrained_gguf(
    "clincoo-qwen3.6-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)

In [ ]:
# 9) Pasang di Ollama (jalankan di mesin tempat Ollama berjalan, bukan di Colab)
# a) Salin folder clincoo-qwen3.6-gguf/*.gguf ke mesin Ollama, lalu buat Modelfile:
#
#    FROM ./clincoo-qwen3.6-q4_k_m.gguf
#    TEMPLATE bawaan Qwen (cukup: ollama show --modelfile qwen3.6 > Modelfile,
#              lalu ganti baris FROM menjadi path .gguf di atas)
#    PARAMETER temperature 0.7
#    PARAMETER top_p 0.8
#    PARAMETER top_k 20
#    PARAMETER num_ctx 4096
#
# b) Buat model baru:
#    ollama create clincoo-ai -f Modelfile
#    ollama run clincoo-ai
#
# c) Di backend labs, ganti nama model 'qwen3.6' menjadi 'clincoo-ai' pada
#    askOllamaDirect() di functions/helpers.js.

## Catatan penting

1. Kurasi dataset sebelum latihan: buang contoh yang salah/kurang berkualitas dari
   `qwen3.6-dataset.jsonl` (data kecil tapi rapi > data banyak yang berisik).
2. `seed-ideal.jsonl` adalah contoh gaya Clincoo buatan model besar, dipakai untuk
   mengunci gaya bahasa dan logika jawaban.
3. Setelah fine-tune, uji banding jawaban model lama vs baru di beberapa pertanyaan
   sampai sebelum dipakai produksi.